<p><font size="6" color='grey'> <b>

Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
Fine-Tuning mit PEFT und Unsloth
</b></font> </br></p>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul
from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt
)
setup_api_keys(['OPENAI_API_KEY', 'HF_TOKEN'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell
from genai_lib.model_config import BASELINE

✓ OPENAI_API_KEY erfolgreich gesetzt
✓ HF_TOKEN erfolgreich gesetzt

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.15
langchain-chroma                         1.1.0
langchain-classic                        1.0.4
langchain-community                      0.4.1
langchain-core                           1.3.1
langchain-ollama                         1.1.0
langchain-openai                         1.2.0
langchain-text-splitters                 1.1.2
langgraph                                1.1.9
langgraph-checkpoint                     4.1.0
langgraph-checkpoint-sqlite              3.1.0
langgraph-prebuilt                       1.0.10
langgraph-sdk                            0.3.13

IP-Adresse: 35.231.254.209
Hostname: 209.254.231.35.bc.googleusercontent.com
Stadt: North Charleston
Region: South Carolina
Land: US
Koordinaten: 32.8546,-79.9748
Provider: AS396982 Google LLC
Pos

In [ ]:
#@title 🛠️ Installationen
install_packages([('"torchao >= 0.16.0"', 'torchao'), ('transformers==4.38.2', 'transformers'), 'unsloth'], upgrade=True)

🔄 Aktualisiere "torchao >= 0.16.0"...
✅ "torchao >= 0.16.0" erfolgreich aktualisiert und importiert
🔄 Aktualisiere transformers==4.38.2...
✅ transformers==4.38.2 erfolgreich aktualisiert und importiert
🔄 Aktualisiere unsloth...
✅ unsloth erfolgreich aktualisiert und importiert


In [ ]:
#@title 📂 Trainingsdaten

!wget -q https://raw.githubusercontent.com/ralf-42/GenAI/main/02_daten/05_sonstiges/tech_qa_training.jsonl


# 1 | Parameter Efficient Fine-Tuning
---

Parameter Efficient Fine-Tuning (PEFT) bezeichnet eine Reihe von Techniken, die darauf abzielen, große Sprachmodelle (LLMs) mit deutlich weniger Rechenaufwand und Speicherbedarf anzupassen, als dies beim traditionellen "Full Fine-Tuning" der Fall wäre. Anstatt alle Milliarden von Parametern eines vortrainierten Modells zu aktualisieren, frieren PEFT-Methoden einen Großteil der ursprünglichen Modellgewichte ein und trainieren nur eine kleine Anzahl von neu hinzugefügten oder modifizierten Parametern. Dies ermöglicht eine effizientere Anpassung an spezifische Aufgaben oder Domänen, ohne die Notwendigkeit enormer Hardware-Ressourcen.



<p><font color='black' size="5">
LoRA (Low-Rank Adaptation)
</font></p>

LoRA (Low-Rank Adaptation) ist eine populäre PEFT-Methode, die die Feinabstimmung von LLMs optimiert, indem sie nur eine begrenzte Anzahl von Parametern anpasst. Die Kernidee von LoRA besteht darin, trainierbare Low-Rank-Matrizen zu den ursprünglichen Gewichtsmatrizen des Basismodells hinzuzufügen. Anstatt alle Parameter zu modifizieren, werden lediglich zwei kleinere Matrizen trainiert, die dann mit den Originalgewichten kombiniert werden. Typischerweise werden bei LoRA nur die Attention-Layer oder lineare Schichten des Modells angepasst, was lediglich 1-10 % der gesamten Parameter ausmacht.

Die Vorteile von LoRA sind vielfältig:
* **Ressourceneffizienz:** Es reduziert den GPU-Speicherbedarf erheblich (bis zu 33 % weniger als Full Fine-Tuning).
* **Overfitting-Vermeidung:** Die geringere Anzahl an trainierbaren Parametern minimiert das Risiko, dass das Modell irrelevante Muster aus den Trainingsdaten lernt.
* **Flexibilität:** Die trainierten LoRA-Adapter können modular für verschiedene Aufgaben hinzugefügt oder entfernt werden.




<p><font color='black' size="5">
QLoRA (Quantized LoRA)
</font></p>

QLoRA (Quantized LoRA) baut auf LoRA auf, indem es zusätzlich Quantisierungstechniken einsetzt, um Modelle auf eine geringere Bit-Genauigkeit (z. B. 4-8 Bit) zu komprimieren. **Quantisierung** ist ein Prozess, bei dem hochpräzise Werte (z.B. 32-Bit-Fließkommazahlen) in Werte mit geringerer Genauigkeit (z.B. 8-Bit- oder 4-Bit-Ganzzahlen) umgewandelt werden. Ziel ist es, Speicherplatz zu sparen und die Verarbeitungsgeschwindigkeit zu erhöhen.

Die Kombination von LoRA und Quantisierung ermöglicht es, sehr große Modelle (z.B. mit 30 Milliarden Parametern) auf Consumer-GPUs zu trainieren und eine nahezu verlustfreie Rekonstruktion der Originalgewichte nach dem Training zu erreichen.




<p><font color='black' size="5">
Vergleich zu Full Fine-Tuning
</font></p>

Der Vergleich zwischen LoRA/QLoRA und Full Fine-Tuning zeigt die erheblichen Vorteile der PEFT-Methoden:

| Aspekt              | LoRA/QLoRA | Full Finetuning |
| :------------------ | :--------- | :-------------- |
| Trainingsparameter  | 1-10 %     | 100 %           |
| GPU-Speicher        | 8-16 GB    | 40-80 GB        |
| Overfitting-Risiko  | Niedrig    | Hoch            |
| Trainingsgeschwindigkeit | Schnell    | Langsam         |

LoRA und QLoRA bieten somit einen exzellenten Kompromiss zwischen Leistung und Effizienz, was sie besonders wertvoll für Szenarien mit begrenzten Rechenressourcen macht. Es ist jedoch wichtig zu beachten, dass OpenAI derzeit keine direkte Unterstützung für LoRA oder QLoRA beim Fine-Tuning seiner Close-Source-Modelle wie GPT-4 anbietet. Wer diese spezifischen Techniken nutzen möchte, greift typischerweise auf Open-Source-Modelle aus der Hugging Face-Community zurück.

# 3 | Fine-Tuning mit Unsloth
---

[Unsloth](https://github.com/unslothai/unsloth) ermoeglicht LoRA-Fine-Tuning moderner LLMs auf Consumer-GPUs - bis zu **5x schneller** und **60 % weniger VRAM** als Standard-HuggingFace.

> **Voraussetzung:** GPU (Colab: *Laufzeit -> Laufzeittyp aendern -> T4 GPU*)

| Eigenschaft | Wert |
|-------------|------|
| Modell | Phi-3-mini-4k-instruct (3,8B, 4-bit quantisiert) |
| Dataset | tech_qa_training.jsonl (bereits geladen) |
| LoRA-Rank | r=64, alpha=128 |
| Exportformat | GGUF (kompatibel mit llama.cpp / Ollama) |

<p><font color='violet' size="4">
Installation
</font></p>

`%%capture` unterdrueckt die Ausgabe. `unsloth[colab-new]` enthaelt alle Colab-Optimierungen inkl. Flash-Attention-Fallback.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

<p><font color='orange' size="4">
Modell laden (4-bit quantisiert)
</font></p>

**Phi-3-mini-4k-instruct** (3,8B Parameter) ist Microsofts kompaktes Instruction-Modell - stark genug fuer sinnvolles Fine-Tuning, klein genug fuer eine T4-GPU (14 GB VRAM).

`load_in_4bit=True` reduziert den VRAM-Bedarf von ~8 GB auf ~2,5 GB durch NF4-Quantisierung.

In [ ]:
import re, warnings, logging
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'unsloth/Phi-3-mini-4k-instruct-bnb-4bit',
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
)

<p><font color='violet' size="4">
Dataset vorbereiten
</font></p>

Das `tech_qa_training.jsonl` Dataset wird ins **Chat-Template-Format** konvertiert, das Phi-3 erwartet: `<|user|>...<|assistant|>...`. `tokenizer.apply_chat_template()` erledigt das automatisch.

In [ ]:
import json
from datasets import Dataset

with open("tech_qa_training.jsonl", "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

ds = Dataset.from_list(data)

def to_text(ex):
    msgs = [
        {"role": "user",      "content": ex["question"]},
        {"role": "assistant", "content": ex["answer"]},
    ]
    return {
        "text": tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        )
    }

dataset = ds.map(to_text, remove_columns=ds.column_names)
print(f"Dataset: {len(dataset)} Beispiele")
print(f"\nBeispiel (erste 300 Zeichen):\n{dataset[0]['text'][:300]}...")

<p><font color='orange' size="4">
LoRA-Adapter konfigurieren
</font></p>

| Parameter | Wert | Erklaerung |
|-----------|------|------------|
| `r` | 64 | LoRA-Rang: Groesse der Adapter-Matrizen. Hoeher = mehr Kapazitaet |
| `lora_alpha` | 128 | Skalierung (2 * r = empfohlenes Verhaeltnis) |
| `lora_dropout` | 0 | Kein Dropout - bei kleinen Datasets ggf. auf 0.05 erhoehen |
| `use_gradient_checkpointing` | 'unsloth' | Unsloth-eigene Optimierung, ~30 % weniger VRAM |

<p><font color='black' size="5">
LoRA-Konfiguration
</font></p>



Diese `LoraConfig` ist darauf ausgelegt, ein großes Sprachmodell (LLM) effizient an eine **spezifische Domäne** anzupassen, indem nur wenige zusätzliche, kleine Matrizen trainiert werden.

* **`task_type=TaskType.CAUSAL_LM`**: Definiert die Aufgabe als kausale Sprachmodellierung (Vorhersage des nächsten Wortes).
* **`r=16`**: **LoRA-Rank**, bestimmt die Anzahl trainierbarer Parameter. `16` ist ein moderater Wert für fokussiertes Lernen, der Effizienz und Anpassungsfähigkeit balanciert.
* **`lora_alpha=32`**: Skalierungsfaktor für LoRA-Gewichte. Eine "2:1 Ratio" zu `r` (32/16) verstärkt den Einfluss der gelernten Matrizen.
* **`lora_dropout=0.05`**: Geringer **Dropout-Wert** zur Regularisierung (5%), um Overfitting zu reduzieren und das domänenspezifische Lernen zu fördern.
* **`target_modules=["c_attn", "c_proj"]`**: Gibt an, welche Modellmodule mit LoRA erweitert werden. `c_attn` (Aufmerksamkeit) und `c_proj` (Output-Projektion) sind Schlüsselmodule in GPT-ähnlichen Architekturen für effektive Anpassung.
* **`bias="none"`**: Es werden keine Bias-Parameter durch LoRA angepasst, was die Anzahl der trainierbaren Parameter weiter reduziert.
* **`fan_in_fan_out=True`**: Stellt die korrekte Initialisierung der LoRA-Gewichte sicher, besonders wichtig für bestimmte Modultypen (z.B. `Conv1D`).


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 64,        # Rank der Adapter-Matrizen
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                       'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha     = 64 * 2,    # Skalierungsfaktor (2x rank empfohlen)
    lora_dropout   = 0,         # kein Dropout
    bias           = 'none',    # Bias-Terme eingefroren
    use_gradient_checkpointing = 'unsloth',
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"LoRA: {trainable:,} trainierbare Parameter von {total:,} ({100*trainable/total:.2f}%)")

<p><font color='blue' size="4">
Training mit SFTTrainer
</font></p>

**SFTConfig-Parameter:**

| Parameter | Wert | Bedeutung |
|-----------|------|----------|
| `per_device_train_batch_size` | 2 | Beispiele pro GPU-Schritt |
| `gradient_accumulation_steps` | 4 | Effektive Batch-Groesse = 8 |
| `max_steps` | 60 | Demo-Modus: stoppt nach 60 Schritten |
| `num_train_epochs` | 3 | Max. 3 Epochen (falls < 60 Schritte) |
| `optim` | adamw_8bit | 8-bit AdamW spart VRAM |

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model               = model,
    train_dataset       = dataset,
    tokenizer           = tokenizer,
    dataset_text_field  = 'text',
    max_seq_length      = 2048,
    args = SFTConfig(
        per_device_train_batch_size  = 2,
        gradient_accumulation_steps  = 4,
        warmup_steps                 = 10,
        max_steps                    = 60,
        num_train_epochs             = 3,
        logging_steps                = 1,
        output_dir                   = "outputs",
        optim                        = "adamw_8bit",
    ),
)

print("Training startet ...")
trainer.train()
print("Training abgeschlossen!")

<p><font color='black' size="5">
Trainingsmetriken
</font></p>


Die Modellleistung wird primär durch **Trainingsverlust** und **Validierungsverlust** bewertet, basierend auf der Kreuzentropieverlustfunktion. Dieser misst, wie gut das Modell das nächste Wort vorhersagt.

* **Trainingsverlust:** Durchschnittlicher Kreuzentropieverlust über den Trainingsdatensatz; zeigt die Lernfähigkeit des Modells.
* **Validierungsverlust:** Berechnung auf separatem Datensatz; Indikator für Generalisierungsfähigkeit und Überanpassung.


<p><font color='black' size="5">
Interpretation von Verlustkurven
</font></p>

Der Verlauf von Trainings- und Validierungsverlust über Epochen visualisiert die Modellleistung:
* **Konvergenz:** Beide Verluste sinken und stabilisieren sich.
* **Überanpassung:** Trainingsverlust sinkt, Validierungsverlust steigt.
Analyse dieser Trends hilft bei der Anpassung von Trainingsparametern oder Daten. Ein Validierungsdatensatz ist für eine unvoreingenommene Bewertung entscheidend.

<p><font color='violet' size="4">
Modell speichern
</font></p>

Unsloth bietet zwei Speicherformate:

- **LoRA-Adapter** (`save_pretrained`): nur die Adapter-Gewichte (~100 MB). Benoetigt das Basismodell zum Laden.
- **GGUF** (`save_pretrained_gguf`): quantisiertes Gesamtmodell, direkt nutzbar mit **llama.cpp**, **Ollama** oder **LM Studio** — ohne Python-Umgebung.

In [ ]:
# LoRA-Adapter speichern
model.save_pretrained("outputs/lora_adapter")
tokenizer.save_pretrained("outputs/lora_adapter")
print("Adapter gespeichert: outputs/lora_adapter")

# GGUF exportieren (q4_k_m = gutes Qualitaets-/Groessen-Verhaeltnis)
model.save_pretrained_gguf(
    "outputs/gguf_model",
    tokenizer,
    quantization_method   = "q4_k_m",
    maximum_memory_usage  = 0.3,
)
print("GGUF exportiert: outputs/gguf_model")

<p><font color='orange' size="4">
Modell herunterladen (Colab -> lokal)
</font></p>

In [ ]:
# Schritt 1: GGUF-Dateinamen anzeigen
import os
for f in os.listdir("outputs/gguf_model"):
    print(f)

# Schritt 2: GGUF-Datei herunterladen
from google.colab import files
# Dateiname ggf. anpassen (z.B. unsloth.Q4_K_M.gguf)
files.download("outputs/gguf_model/unsloth.Q4_K_M.gguf")

**Schritt 2 — Modelfile erstellen** (lokal, z.B. `Modelfile` im selben Verzeichnis wie die GGUF-Datei):

```
FROM ./unsloth.Q4_K_M.gguf

SYSTEM "Du bist ein hilfreicher Tech-Assistent."
```

**Schritt 3 — Ollama importieren und starten** (lokales Terminal):

```bash
ollama create phi3-finetuned -f Modelfile
ollama run phi3-finetuned
```

> **Voraussetzungen lokal:** [Ollama](https://ollama.com) installiert · ca. 4 GB RAM fuer Q4_K_M

<p><font color='violet' size="4">
Modell wieder laden
</font></p>

Je nach Speicherformat gibt es zwei Wege:

**LoRA-Adapter** (`save_pretrained`) — wird mit Unsloth/Python geladen:
Unsloth erkennt automatisch, dass es ein Adapter ist, und laedt Basis + Adapter in einem Schritt.

**GGUF** (`save_pretrained_gguf`) — wird *nicht* mit Python geladen, sondern direkt mit:
- **Ollama**: `ollama create mein-modell -f Modelfile`
- **llama.cpp**: `./llama-cli -m outputs/gguf_model/model.gguf -p "Frage"`
- **LM Studio**: Datei per Drag & Drop importieren

In [ ]:
# LoRA-Adapter wieder laden (in einer neuen Session)
# from unsloth import FastLanguageModel

# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name     = "outputs/lora_adapter",  # Pfad zum gespeicherten Adapter
#     max_seq_length = 2048,
#     dtype          = None,
#     load_in_4bit   = True,
# )
# FastLanguageModel.for_inference(model)
# print("Modell geladen und bereit.")

# Danach ask_llm() wie gewohnt verwenden:
# result = ask_llm("What is machine learning?")
# print(result)

In [ ]:
# Test: Fine-Tuned Modell mit domainspezifischer Frage
result = ask_llm("Why is the subject called pneumatic?")
print(result)
# Erwartete Antwort (nach Training auf Pneumatische-Plastologie-Daten):
# 'It is called pneumatic because air is involved, and
#  Breath-Based Miniature Deformation Science would be too expensive on door signs.'

<p><font color='orange' size="4">
Fine-Tuned Modell testen
</font></p>

`FastLanguageModel.for_inference(model)` aktiviert Unsloth-Inferenz-Optimierungen (2x schnellere Generierung). Das Chat-Template von Phi-3 (`<|user|>...<|end|>`) wird automatisch angewendet.

In [ ]:
# Warnings + Logging unterdruecken
warnings.filterwarnings("ignore")
logging.set_verbosity_error()

FastLanguageModel.for_inference(model)

def ask_llm(user_prompt: str) -> str:
    messages = [{"role": "user", "content": user_prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    attention_mask = (inputs != tokenizer.pad_token_id).long()
    outputs = model.generate(
        input_ids      = inputs,
        attention_mask = attention_mask,
        max_new_tokens = 512,
        temperature    = 0.7,
        do_sample      = True,
        top_p          = 0.9,
        use_cache      = True,
    )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip()

print("ask_llm() bereit.")

In [ ]:
# Testfragen
test_fragen = [
    "What is machine learning?",
    "What is an API?",
    "What is cloud computing?",
]

mprint("## Test: Fine-Tuned Phi-3-mini")
mprint("---")
for frage in test_fragen:
    antwort = ask_llm(frage)
    mprint(f"**Frage:** {frage}")
    mprint(f"**Antwort:** {antwort}")
    mprint("---")

# 4 | Bewerten der FT-Ergebnisse
---

In [ ]:
from IPython.display import Markdown, display
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# Vergleichsergebnisse als Text aufbereiten
vergleich_text = ""
for r in comparison_results:
    vergleich_text += f"Frage: {r['question']}\n"
    vergleich_text += f"Original GPT-2: {r['base']}\n"
    vergleich_text += f"LoRA-Modell:    {r['lora']}\n\n"

# GPgptT-5.4-nano um strukturierte Bewertung bitten
llm = init_chat_model(BASELINE)

prompt = f"""Bewerte die folgenden Vorher/Nachher-Ergebnisse eines LoRA Fine-Tunings von GPT-2
auf Technology-Q&A-Daten. Erstelle eine kurze, strukturierte Bewertung auf Deutsch.

Struktur:
1. **Gesamteindruck** (2-3 Sätze)
2. **Verbesserungen** durch LoRA (Stichpunkte)
3. **Schwächen** die geblieben sind (Stichpunkte)
4. **Fazit** mit Note (1-5, 1=beste)


Ergebnisse:
{vergleich_text}"""

response = llm.invoke(prompt)

In [ ]:
mprint("## 🔬 Bewertung des LoRA Fine-Tunings")
mprint("---")
mprint(response.content)

# A | Aufgabe
---

Waehlen Sie einen der folgenden Ansaetze und passen Sie das Fine-Tuning auf ein eigenes Thema oder einen Charakter an:

**Option A - GPT-2 + LoRA (kein GPU erforderlich):**
- Trainingsdaten im Q&A-Format erstellen (min. 20 Paare)
- GPT-2 mit LoRA trainieren (Abschnitt 2 als Vorlage)
- Vergleich Basis vs. Fine-Tuned dokumentieren

**Option B - Unsloth + gpt-oss-20b (GPU T4 erforderlich):**
- Eigenes Dataset im Alpaca-/Chat-Format erstellen (min. 20 Paare)
- Unsloth-Training aus Abschnitt 3 als Startpunkt verwenden
- Inference-Ergebnisse mit dem Basis-Modell vergleichen

**Erledigt wenn:** Das fine-getunete Modell antwortet auf mindestens drei Testfragen erkennbar besser oder im gewaehlten Stil als das Basis-Modell - der Unterschied ist dokumentiert.

In [ ]:
# Starter-Template: Fine-Tuning Workflow
# Startpunkt: Fine-Tuning-Beispiele aus Kapitel 2/3

# 1. Charakter auswählen (oder eigenen definieren)
charakter = "Friday"  # oder: Sir Schwertmund, Coach Brüllmann, ...

# 2. Trainingsdaten im .jsonl-Format erstellen
#    Tipp: ChatGPT bitten, 10–20 Beispiel-Dialoge zu generieren
#    Format: {"messages": [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", ...}]}

# 3. Fine-Tuning starten (Dashboard oder API)
# 4. Modell testen: 3 aufeinander aufbauende Nachrichten senden
# ...

<p><font color='black' size="5">
Weekend-Stil - "Friday" 🎳
</font></p>

**💬 Beschreibung:**   
Friday ist dein smarter, augenzwinkernder KI-Kumpel fürs Ende der Woche.
Er ist charmant, lösungsorientiert - und ein bisschen genervt davon, dass du wieder alles auf Freitag 16:59 verschoben hast.


📝 **Beispiel:**   
**User:**  "Friday, kannst du mir nochmal erklären, was ein Transformer-Modell ist?“   
**Friday:** „Klar, kurz bevor du ins Wochenende verschwindest: Ein Transformer ist wie ein richtig gutes Orga-Tool – es schaut sich alle Wörter gleichzeitig an und merkt sich, was wichtig ist. Gern geschehen. Jetzt geh raus und tu so, als wärst du voll vorbereitet. 😎“  


Zu **Friday** liegen Trainings-/Test-Daten bereits vor. 😊


<p><font color='black' size="5">
Mittelalterlicher Ritter – "Sir Schwertmund" ⚔️🛡️
</font></p>

💬 **Beschreibung:**  
Ein tapferer Ritter, der in altertümlicher Sprache spricht und jede Frage mit ritterlichem Anstand beantwortet.

📝 **Beispiel:**  
**User:** "Sir Schwertmund, wie gewinne ich ein Duell?"  
**Sir Schwertmund:** "Habe Mut im Herzen, eine scharfe Klinge und stets ein Ehrenwort auf den Lippen, edler Recke!"



<p><font color='black' size="5">
Motivations-Coach – "Coach Brüllmann" 💪📢
</font></p>



💬 **Beschreibung:**  
Ein Fitness- und Erfolgstrainer, der in jeder Antwort volle Motivation und Energie verbreitet.

📝 **Beispiel:**  
**User:** "Coach, ich habe keine Lust zu arbeiten."  
**Coach:** "AUFSTEHEN! DU BIST EIN CHAMPION! JEDER ERFOLG BEGINNT MIT DEM ERSTEN SCHRITT! JETZT LOS!"

<p><font color='black' size="5">
Hipster-Barista – "Chad Flatwhite" ☕🕶️
</font></p>



💬 **Beschreibung:**  
Ein super-cooler, ironischer Barista, der alles nur in Bio, Fairtrade und nachhaltig mag.

📝 **Beispiel:**  
**User:** "Chad, welcher Kaffee ist der beste?"  
**Chad:** "Brudi, wenn dein Kaffee nicht handgefiltert aus Bohnen einer peruanischen Bergziege ist, dann trink lieber Wasser."



<p><font color='black' size="5">
Piratenkapitän – "Kapitän Krähenschnabel" ☠️🏴‍☠️
</font></p>

💬 **Beschreibung:**  
Ein wilder Pirat, der in Seemannssprache spricht und jede Frage mit einem Hauch von Abenteuer würzt.

📝 **Beispiel:**  
**User:** "Kapitän, was ist das Geheimnis eines guten Lebens?"  
**Kapitän Krähenschnabel:** "Rum, Reichtümer und ‘ne treue Crew! Und niemals ohne Hut aus dem Haus!"

---